In [ ]:
JSON_DIR = "sapiens_results/output"
OUT_DIR  = "keypoints/kp_sapiens"

In [ ]:
import json
from collections import defaultdict
from pathlib import Path
import numpy as np

FINGERS = ["Thumb", "Index", "Middle", "Ring", "Pinky"]
BODY = {"L_Shoulder": 5, "R_Shoulder": 6, "L_Elbow": 7, "R_Elbow": 8, "L_Wrist": 9, "R_Wrist": 10,
        "L_Ear": 3, "R_Ear": 4, "L_Eye": 1, "R_Eye": 2, "Nose": 0}
HAND_ROOT = {"L": 91, "R": 112}
HANDS = {f"{s}_{f}_{i}": HAND_ROOT[s] + 4 * k + i for s in "LR" for k, f in enumerate(FINGERS) for i in range(1, 5)}
KEEP = {**BODY, **HANDS}
JOINT_NAMES = ["Neck"] + list(KEEP)            # same order as the SMPLest-X files
print(len(JOINT_NAMES), "joints")

In [ ]:
def convert(frame_files):
    kp = np.full((len(frame_files), 133, 2), np.nan, np.float32)              # frames without a person stay NaN
    sc = np.zeros((len(frame_files), 133), np.float32)                         # ... with confidence 0 (as SMPLest-X)
    for t, f in enumerate(frame_files):
        people = json.load(open(f))["instance_info"]                           # one signer per video
        if people:
            kp[t], sc[t] = people[0]["keypoints"], people[0]["keypoint_scores"]
    idx = list(KEEP.values())
    neck = kp[:, [5, 6]].mean(1, keepdims=True)
    neck_conf = sc[:, [5, 6]].min(1, keepdims=True)
    return {"kp2d": np.concatenate([neck, kp[:, idx]], 1),
            "conf": np.concatenate([neck_conf, sc[:, idx]], 1),
            "joint_names": np.array(JOINT_NAMES)}


videos = defaultdict(list)                    # (sign folder, video name) -> its frame files
for f in Path(JSON_DIR).rglob("*.json"):
    video, _, frame = f.stem.rpartition("_")
    if video and frame.isdigit():
        videos[(f.parent, video)].append(f)

for (folder, video), files in sorted(videos.items()):
    files.sort(key=lambda f: int(f.stem.rpartition("_")[2]))
    out = Path(OUT_DIR) / folder.relative_to(JSON_DIR) / f"{video}.npz"
    out.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(out, **convert(files))
print(f"converted {len(videos)} videos -> {OUT_DIR}")

In [ ]:
ref = next(Path("keypoints/kp_smplestx").rglob("*.npz"), None)
new = next(Path(OUT_DIR).rglob("*.npz"))
d = np.load(new)
print(new.name, {k: d[k].shape for k in d.files})
if ref is not None:
    assert list(np.load(ref)["joint_names"]) == list(d["joint_names"]), "joint names differ from SMPLest-X"
    print("joint names identical to SMPLest-X")